In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
import random


class CategoriaInvalidaException(Exception):
    pass

class PistaInvalidaException(Exception):
    pass

class VotoInvalidoException(Exception):
    pass

class FaseJuego(Enum):
    REGISTRO = "Registro"
    ASIGNACION = "Asignación"
    PISTAS = "Pistas"
    DEBATE = "Debate"
    VOTACION = "Votación"
    RESULTADO = "Resultado"

class EstadoJugador(Enum):
    ACTIVO = "Activo"
    ELIMINADO = "Eliminado"

class Ganador(Enum):
    CIVILES = "Civiles"
    IMPOSTOR = "Impostor"

class ConfiguracionPartida:
    def __init__(self, num_rondas, tiempo_debate, categoria,
                 ultima_oportunidad, longitud_maxima_pista):
        self.num_rondas = num_rondas
        self.tiempo_debate = tiempo_debate          # medido en segundos
        self.categoria = categoria
        self.ultima_oportunidad = ultima_oportunidad
        self.longitud_maxima_pista = longitud_maxima_pista

    def validar_configuracion(self):
        if self.num_rondas <= 0:
            raise ValueError("Debe haber al menos una ronda de juego.")

        if self.tiempo_debate <= 5:
            raise ValueError("Debe haber tiempo para debatir.")


class Rol(ABC):
    def __init__(self, nombre):
        self._nombre = nombre

    @property
    def nombre(self):
        return self._nombre

    @abstractmethod
    def conoce_palabra(self):
        pass

    @abstractmethod
    def obtener_palabra_visible(self):
        pass


class Civil(Rol):
    def __init__(self, palabra_secreta):
        super().__init__("Civil")
        self._palabra_secreta = palabra_secreta

    def conoce_palabra(self):
        return True

    def obtener_palabra_visible(self):
        return self._palabra_secreta.texto

class Impostor(Rol):
    def __init__(self):
        super().__init__("Impostor")

    def conoce_palabra(self):
        return False

    def obtener_palabra_visible(self):
        return "???"

    def intentar_adivinar(self, intento, palabra_secreta):
        return palabra_secreta.coincide_con(intento)


class Jugador:
    def __init__(self, id, nombre):
        self._id = id
        self._nombre = nombre
        self._rol = None
        self._eliminado = False

    @property
    def id(self):
        return self._id

    @property
    def nombre(self):
        return self._nombre

    @property
    def eliminado(self):
        return self._eliminado

    def eliminar(self):
        self._eliminado = True

    def get_rol(self):
        return self._rol

    def set_rol(self, rol):
        self._rol = rol

    def dar_pista(self, texto, numero_turno):
        return Pista(self, texto, numero_turno)

    def votar(self, acusado):
        return Voto(self, acusado)


class PalabraSecreta:
    def __init__(self, texto, categoria):
        self._texto = texto
        self._categoria = categoria

    @property
    def texto(self):
        return self._texto

    @property
    def categoria(self):
        return self._categoria

    def coincide_con(self, texto):
        return self._texto.strip().lower() == texto.strip().lower()


class Categoria:
    def __init__(self, tipo):
        self._tipo = tipo
        self._palabras = []

    @property
    def tipo(self):
        return self._tipo

    @property
    def palabras(self):
        return self._palabras

    def agregar_palabra(self, palabra):
        if palabra not in self._palabras:
            self._palabras.append(palabra)

    def eliminar_palabra(self, palabra):
        if palabra in self._palabras:
            self._palabras.remove(palabra)


class Pista:
    def __init__(self, autor, texto, numero_turno):
        self._autor = autor
        self._texto = texto
        self._numero_turno = numero_turno

    def get_texto(self):
        return self._texto

    def get_autor(self):
        return self._autor

    @property
    def numero_turno(self):
        return self._numero_turno


class Voto:
    def __init__(self, votante, acusado):
        if votante is acusado:
            raise VotoInvalidoException("Un jugador no puede votar por sí mismo, por favor elige otro jugador.")
        self._votante = votante
        self._acusado = acusado

    def get_votante(self):
        return self._votante

    def get_acusado(self):
        return self._acusado


class ResultadoPartida:
    def __init__(self, ganador, motivo, impostor, palabra_secreta, ganadores):
        self.ganador = ganador
        self.motivo = motivo
        self.impostor = impostor
        self.palabra_secreta = palabra_secreta
        self.ganadores = ganadores      # jugadores que ganaron la partida


class Ronda:
    _ORDEN_FASES = [FaseJuego.PISTAS, FaseJuego.DEBATE,
                    FaseJuego.VOTACION, FaseJuego.RESULTADO]

    def __init__(self, numero, jugadores_activos, configuracion):
        self._numero = numero
        self._jugadores_activos = list(jugadores_activos)
        self._configuracion = configuracion
        self._pistas = []
        self._votos = []
        self._fase = FaseJuego.PISTAS
        self.eliminado = None      
        self.doble_eliminacion = False      

    @property
    def numero(self):
        return self._numero

    @property
    def pistas(self):
        return list(self._pistas)

    @property
    def votos(self):
        return list(self._votos)

    @property
    def fase(self):
        return self._fase

    @property
    def siguiente_turno(self):
        return len(self._pistas) + 1

    def registrar_pista(self, pista):
        if self._fase != FaseJuego.PISTAS:
            raise PistaInvalidaException("Solo se pueden dar pistas en la fase de pistas.")
        autor = pista.get_autor()
        if autor not in self._jugadores_activos:
            raise PistaInvalidaException("Solo los jugadores activos pueden dar pistas.")
        if any(p.get_autor() is autor for p in self._pistas):
            raise PistaInvalidaException("Cada jugador solo puede dar una pista por ronda.")
        texto = pista.get_texto().strip()
        if not texto:
            raise PistaInvalidaException("La pista no puede estar vacía.")
        self._pistas.append(pista)

    def registrar_voto(self, voto):
        if self._fase != FaseJuego.VOTACION:
            raise VotoInvalidoException("Solo se puede votar en la fase de votación.")
        votante = voto.get_votante()
        if any(v.get_votante() is votante for v in self._votos):
            raise VotoInvalidoException("Solo se puede votar una vez por ronda")
        self._votos.append(voto)

    def avanzar_fase(self):
        posicion = self._ORDEN_FASES.index(self._fase)
        if posicion == len(self._ORDEN_FASES) - 1:
            raise ValueError("La ronda ya está en su última fase.")
        self._fase = self._ORDEN_FASES[posicion + 1]


class Partida:
    def __init__(self, jugadores, palabra_secreta, configuracion,asignador, contador_votos, validadores_pista):
      if not 3 <= len(jugadores) <= 10:
            raise ValueError("Debe haber entre 3 y 10 jugadores")
      nombres = [j.nombre.strip().lower() for j in jugadores]
      if len(set(nombres)) != len(nombres):
        raise ValueError("Los nombres de los jugadores no se pueden repetir")
      self._jugadores = list(jugadores)
      self._rondas = []
      self._palabra_secreta = palabra_secreta
      self._configuracion = configuracion
      self._asignador = asignador
      self._contador_votos = contador_votos
      self._validadores_pista = validadores_pista

    @property
    def jugadores(self):
        return list(self._jugadores)

    @property
    def rondas(self):
        return list(self._rondas)

    @property
    def ronda_actual(self):
        return self._rondas[-1] if self._rondas else None

    @property
    def palabra_secreta(self):
        return self._palabra_secreta

    @property
    def jugadores_activos(self):
        return [a for a in self._jugadores if not a.eliminado]

    def nueva_ronda(self):
        ronda = Ronda(len(self._rondas) + 1, self.jugadores_activos, self._configuracion)
        self._rondas.append(ronda)
        return ronda

    def iniciar(self):
      self._configuracion.validar_configuracion()
      self._asignador.asignar(self._jugadores, self._palabra_secreta)
      self._rondas = []
      return self.nueva_ronda()

    def avanzar_fase(self):
        self.ronda_actual.avanzar_fase()

    def dar_pista(self, jugador, texto):
      ronda = self.ronda_actual
      pista = jugador.dar_pista(texto, ronda.siguiente_turno)
      for validador in self._validadores_pista:
          validador.validar(pista)
      ronda.registrar_pista(pista)
      return pista

    def votar(self, votante, acusado):
        voto = votante.votar(acusado)
        self.ronda_actual.registrar_voto(voto)
        return voto

    def cerrar_votacion(self):
        ronda = self.ronda_actual
        acusado = self._contador_votos.contar(ronda.votos)
        if acusado is not None:
            acusado.eliminar()
            ronda.eliminado = acusado
        return acusado

    def _crear_resultado(self, ganador, motivo, impostor):
        if ganador == Ganador.IMPOSTOR:
            ganadores = [impostor]
        else:
            ganadores = [a for a in self._jugadores if a is not impostor]
        return ResultadoPartida(ganador, motivo, impostor, self._palabra_secreta, ganadores)

    def determinar_ganador(self, intento_impostor=None):
        impostor = next(j for j in self._jugadores if isinstance(j.get_rol(), Impostor))
        if impostor.eliminado:
            if (self._configuracion.ultima_oportunidad and intento_impostor
                    and impostor.get_rol().intentar_adivinar(intento_impostor, self._palabra_secreta)):
                return self._crear_resultado(Ganador.IMPOSTOR, "El impostor adivinó la palabra secreta.", impostor)
            return self._crear_resultado(Ganador.CIVILES, "Descubrieron al impostor.", impostor)
        if len(self.jugadores_activos) <= 2:
            return self._crear_resultado(Ganador.IMPOSTOR, "Solo quedan el impostor y un civil.", impostor)
        if len(self._rondas) >= self._configuracion.num_rondas:
            return self._crear_resultado(Ganador.IMPOSTOR, "El impostor sobrevivió todas las rondas.", impostor)
        return None